# Day 12 — End-to-end answer evaluation and regression tests: grade the answer, not only the retrieval

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-12/LEARN.md`](../docs/day-12/LEARN.md).*

Days 07 to 11 measured retrieval: is a labelled chunk in the top 3, or in the context the reader gets? By that measure the pipeline is nearly done. On dev, a labelled chunk is in the context for 107 of 112 answerable questions. But only 69 of them get a correct answer. The rest are lost after retrieval: refused by a rule, copied from the wrong sentence, or blocked by the citation check. A retrieval metric can't see any of that.

This notebook grades **answers**:

1. **A documented rubric**: whether the answer is correct (against each question's facts), grounded (supported by the sources it cites), cited (does it cite a chunk that holds the answer), whether a refusal was right, and **the stage that failed**: retrieval, the gate, the names rule, the reader, or the citation check.
2. **The grader is graded too**: its verdicts against the author's reading of every answer, with every disagreement listed.
3. **The series' recorded failures become regression tests**: 29 cases, from Day 01's "new dads" to Day 11's unchecked neighbour lookup.
4. **One change, decided by answers**: Day 10's names rule (measured and not adopted on Day 10), compared with the current pipeline question by question on **76 new held-out questions** (`questions-v4`).

**Every rule was committed before anything scored a held-out question.** [`evals/preregistration/day-12.md`](../evals/preregistration/day-12.md), the first commit on today's branch together with the new questions, holds the current pipeline, the candidate, the rubric and its known limits, the value of an answer, the dev values, what would be reported and what the decision needs. The notebook recomputes the dev values and stops if they differ.

Everything runs locally: no API key, no new packages, no new model (Day 04's), about a minute. An optional live run grades a hosted model with the same rubric (section 10, off by default).

## 0. Setup

No new packages. The shared code in [`src/rag_from_scratch/`](../src/rag_from_scratch/) gains three things today, because this notebook reuses them:

- `contexts.py`: **Day 11's** documents' structure, section expansion and the end-to-end steps fed a chosen context (`answer_context()`: Day 07's gate on the dense top score, the budget, a reader, the citation check), with the sentence cache and `why_refused()`;
- `queries.py`: **Day 10's** revised names rule (`unknown_names_v2`) and `NamesReader`, the candidate;
- `chat.py`: **Day 06's** prompt and language-model client, for the optional live run.

`tests/test_shared_code.py` checks each against the notebook it came from. `COST` is the pre-registered value of a wrong answer: it costs twice what a correct one gains.

In [1]:
import dataclasses
import hashlib
import json
import math
import os
import platform
import re
import statistics
import sys
import time
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
    import tokenizers
except ImportError as error:
    raise ImportError("Day 12 uses Day 04's packages (numpy, onnxruntime, tokenizers, huggingface-hub). From the "
                      "repository root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.answering import (DIRECTORY, MAX_CONTEXT_WORDS, SENTENCE, TOP_K, ExtractiveReader,  # Day 06's
                                        FixedReply, GeneratorError, Pipeline, answer_units, unknown_names)  # pipeline
from rag_from_scratch.bm25 import BM25Index                              # Day 01's BM25
from rag_from_scratch.chat import PRICES, build_messages, chat_model_from_env   # Day 06's prompt and client (new)
from rag_from_scratch.chunking import section_chunks                     # Day 03's chunker
from rag_from_scratch.contexts import (DocumentStructure, SentenceCache, answer_context, expand_sections,  # Day 11's
                                       why_refused)                                                         # (new)
from rag_from_scratch.embeddings import Embedder                         # Day 04's model
from rag_from_scratch.evaluation import (TIE, compare, first_relevant, hit, load_queries, measure,  # Day 07's harness
                                         outcome, sign_test, tie_margin, top_score, validate, wilson)
from rag_from_scratch.fusion import ALPHA, weighted                      # Day 08's fusion
from rag_from_scratch.index import VectorIndex, sync                     # Day 05's index
from rag_from_scratch.ingest import ingest                               # Day 02's pipeline
from rag_from_scratch.queries import (VOCABULARY, NamesReader, defined_abbreviations, expand,  # Day 10's map and
                                      unknown_names_v2)                                       # names rule (new)

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__} | "
      f"tokenizers {tokenizers.__version__}")

Python 3.13.18 | numpy 2.4.6 | onnxruntime 1.31.0 | tokenizers 0.23.3


In [2]:
K_VALUES = (1, 3, 5, 10)
EVALS = REPO_ROOT / "evals"
V3 = {"questions": EVALS / "questions-v3.jsonl", "manifest": EVALS / "manifest-v3.json"}    # Day 11's 190 queries
V4 = {"questions": EVALS / "questions-v4.jsonl", "manifest": EVALS / "manifest-v4.json"}    # + today's 76 new ones
DAY11_BASELINE = EVALS / "baselines" / "day-11-retrieval-v3.json"   # Day 11's accepted run: retrieval's reference
BASELINE_FILE = EVALS / "baselines" / "day-12-answers-v4.json"      # today's accepted run: answers, graded
REVIEWS_FILE = EVALS / "reviews" / "day-12-answers.jsonl"           # the author's reading of every answer
REGRESSIONS_FILE = EVALS / "regressions.jsonl"                      # the series' recorded failures, as test cases
PREREGISTRATION = EVALS / "preregistration" / "day-12.md"           # today's rules, committed before any held-out run
OUT_DIR = REPO_ROOT / "outputs" / "day-12"                          # this run's results; outputs/ is git-ignored

HITS = TOP_K                  # 3: the current pipeline sends the weighted fusion's top 3 (Day 11 kept it)
BUDGET = MAX_CONTEXT_WORDS    # 300 words, whole chunks (Day 06)
GATE = 0.62                   # Day 07's measured gate on the dense top score (evals/baselines/day-07-retrieval.json)
READER_THRESHOLD = 0.63       # Day 07's measured reader threshold (same file)
SUPPORT_MIN = 0.75            # "grounded": every sentence has at least this share of its words in its cited sources
COST = 2                      # a wrong answer costs twice what a correct one gains (pre-registered assumption)
PIPELINES = ("current", "names-v2")

# What evals/preregistration/day-12.md commits to. Sections 3.1 and 5 recompute these on dev and stop if they differ.
PREREGISTERED = {"dev": {"current": (69, 12, 31, 29, 19, 7), "names-v2": (84, 18, 10, 28, 20, 8)},
                 "better": 15, "worse": 7, "delta": {1: 8, 2: 1, 3: -6}, "frozen": "2026-11-11 07:53:09 UTC"}
prereg = PREREGISTRATION.read_text(encoding="utf-8")
assert "| current pipeline | 69 / 12 / 31 | 29 / 19 | 7 |" in prereg and \
    "| candidate (Day 10's names rule) | 84 / 18 / 10 | 28 / 20 | 8 |" in prereg and \
    "answerable correct **+1**, refused **0**, wrong **−2**" in prereg, "PREREGISTERED must match the pre-registration"

## 1. The same queries, the same snapshot, Day 11's six retrievers

Before changing anything, this section loads Day 11's 190 labelled queries (`questions-v3`), checks the documents against their snapshot, rebuilds Day 11's six retrievers (BM25, dense retrieval, the weighted fusion, the two map retrievers and Day 11's section expansion, now from `src/`) and checks that they **reproduce Day 11's accepted baseline**, query by query, including every section the expansion sent.

In [3]:
manifest3 = json.loads(V3["manifest"].read_text(encoding="utf-8"))
queries3 = load_queries(V3["questions"])
documents, _ = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
by_cid = {c.chunk_id: c for c in chunks}
problems3 = validate(queries3, chunks, manifest3)
v3_sha256 = hashlib.sha256(V3["questions"].read_bytes()).hexdigest()
if v3_sha256 != manifest3["questions_sha256"]:
    problems3.insert(0, f"{V3['questions'].name} changed but its manifest didn't")
print(f"{len(queries3)} labelled queries (version {manifest3['version']}) | {len(chunks)} chunks | label problems: "
      f"{len(problems3)}" + "".join(f"\n  {p}" for p in problems3))
assert not problems3, "The labels don't match the documents: run the Day 07 notebook and review them first."

embedder = Embedder(REPO_ROOT)     # Day 04's model: downloaded once (133 MB), SHA-256 checked on every run
index = VectorIndex(embedder.model["repo"], embedder.revision, embedder.dimensions)
sync(index, chunks, embedder.embed)   # the snapshot's chunks, embedded now (Day 05's sync into an empty index)
rag = Pipeline(index, embedder)       # Day 06's retrieve(): as the user, permission filter before the ranking
bm25 = {user: BM25Index({c.chunk_id: c.text for c in chunks if set(c.allowed_groups) & groups})
        for user, groups in DIRECTORY.items()}
ENTRIES = VOCABULARY + defined_abbreviations(chunks)    # Day 10's reviewed map (measured, not adopted)
structure = DocumentStructure(chunks)                   # Day 11: each document's sections, in order
NAMES = ["BM25", "dense", "weighted", "BM25+map", "weighted+map"]   # Day 10's five retrievers
rankings = {name: {} for name in NAMES}      # retriever -> query id -> FULL ranking, (chunk_id, score) best first


def add_rankings(qs) -> None:
    for q in qs:
        lexical = [(h.id, h.score) for h in bm25[q.user].search(q.question, len(chunks))]
        dense = [(r.id, r.score) for r in rag.retrieve(q.user, q.question, len(chunks))]
        mapped = [(h.id, h.score) for h in bm25[q.user].search(expand(q.question, ENTRIES).text, len(chunks))]
        rankings["BM25"][q.id], rankings["dense"][q.id], rankings["BM25+map"][q.id] = lexical, dense, mapped
        rankings["weighted"][q.id] = weighted(lexical, dense, ALPHA)     # the default retriever since Day 08
        rankings["weighted+map"][q.id] = weighted(mapped, dense, ALPHA)


def neighbours_of(q):
    # Day 11's section expansion (not adopted): the top 3, then each hit's previous and next section, checked
    return expand_sections(rankings["weighted"][q.id], q.user, q.question, "neighbours", by_cid, structure)


add_rankings(queries3)
print(f"Day 10's five retrievers and Day 11's section expansion on {len(queries3)} queries")

190 labelled queries (version 3) | 44 chunks | label problems: 0
tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified
onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified


Day 10's five retrievers and Day 11's section expansion on 190 queries


In [4]:
day11 = json.loads(DAY11_BASELINE.read_text(encoding="utf-8"))
EXPANDED = "weighted+neighbours"


def entry(name: str, q) -> dict:
    # Day 07's baseline entry: the first labelled chunk's rank, how close it is to a tie, the top 3, the best score
    ranked = rankings[name][q.id]
    return {"first_relevant": first_relevant(ranked, q.answer_chunks) if q.answerable else None,
            "margin": tie_margin(ranked, q.answer_chunks) if q.answerable else None,
            "top3": [cid for cid, _ in ranked[:HITS]], "top_score": top_score(ranked)}


def cutoff_gap(q, n: int) -> float:
    # How close the fused scores at ranks n and n + 1 are: under TIE, another CPU's rounding could swap them
    ranked = rankings["weighted"][q.id]
    return ranked[n - 1][1] - ranked[n][1] if n < len(ranked) else math.inf


def found(sent, q) -> bool:
    return any(cid in q.answer_chunks for cid in sent)


def words(sent) -> int:
    return sum(len(by_cid[cid].text.split()) for cid in sent)


def expansion_entry(q) -> dict:
    # Day 11's entry for the expansion: the position of the first labelled section, the sections and words sent, and
    # whether the size-matched control (the weighted fusion's top m) found the answer
    sent = neighbours_of(q).sent
    control = tuple(cid for cid, _ in rankings["weighted"][q.id][:len(sent)])
    gap, control_gap = cutoff_gap(q, HITS), cutoff_gap(q, len(sent))
    return {"first_relevant": next((i for i, cid in enumerate(sent, 1) if cid in q.answer_chunks), None)
            if q.answerable else None,
            "margin": round(gap, 4) if q.answerable and gap != math.inf else None,
            "top3": list(sent[:HITS]), "top_score": top_score(rankings["weighted"][q.id]),
            "sent": list(sent), "words": words(sent),
            "control_found": found(control, q) if q.answerable else None,
            "control_margin": round(control_gap, 4) if q.answerable and control_gap != math.inf else None}


def context_metrics(selection) -> dict:
    sent = {q.id: neighbours_of(q).sent for q in selection}
    control = {q.id: tuple(cid for cid, _ in rankings["weighted"][q.id][:len(sent[q.id])]) for q in selection}
    return {"n": len(selection), "top 3 in context": sum(found(rankings["weighted"][q.id][:HITS] and
                                                               [c for c, _ in rankings["weighted"][q.id][:HITS]], q)
                                                         for q in selection),
            "in context": sum(found(sent[q.id], q) for q in selection),
            "control in context": sum(found(control[q.id], q) for q in selection),
            "mean words": round(sum(words(sent[q.id]) for q in selection) / len(selection), 1),
            "control mean words": round(sum(words(control[q.id]) for q in selection) / len(selection), 1)}


SELECTIONS3 = {"questions": [q for q in queries3 if q.kind == "question" and q.answerable],
               "lookups": [q for q in queries3 if q.kind == "lookup"]}
V3_TEST = [q for q in queries3 if q.split == "test" and q.answerable]
# Day 11's accepted file, with today's ranks, sections and metrics for its six retrievers in place of its own. The
# settings are the accepted ones; the few that are numbers are checked against the objects used here.
reproduced = {**day11, "queries": {q.id: {**{name: entry(name, q) for name in NAMES}, EXPANDED: expansion_entry(q)}
                                   for q in queries3},
              "metrics": {**{f"{kind}, {name}": {k: round(v, 4) if isinstance(v, float) else v
                                                for k, v in measure(rankings[name], sel).items()}
                             for kind, sel in SELECTIONS3.items() for name in NAMES},
                          **{f"{kind}, {EXPANDED}": context_metrics(sel) for kind, sel in SELECTIONS3.items()},
                          f"v3 test questions, {EXPANDED}": context_metrics(V3_TEST)}}
settings = day11["retrievers"]
assert (settings["BM25"]["k1"], settings["BM25"]["b"]) == (bm25["alice"].k1, bm25["alice"].b) and \
    settings["weighted"]["alpha_dense"] == ALPHA and settings["dense"]["revision"] == embedder.revision and \
    settings[EXPANDED]["budget_words"] == BUDGET and settings[EXPANDED]["hits"] == HITS
changes11, ties11 = compare(reproduced, day11)
same_sent = all(reproduced["queries"][q.id][n]["top3"] == day11["queries"][q.id][n]["top3"]
                for q in queries3 for n in NAMES) and \
    all(reproduced["queries"][q.id][EXPANDED]["sent"] == day11["queries"][q.id][EXPANDED]["sent"] for q in queries3)
print(f"{', '.join(NAMES + [EXPANDED])} against {DAY11_BASELINE.relative_to(REPO_ROOT).as_posix()}: "
      + (f"{len(changes11)} changes" + "".join(f"\n  {c}" for c in changes11) if changes11 else
         f"no changes ({len(queries3)} queries x 6: every rank, best score, section sent and metric; top 3 lists and "
         f"sections sent {'identical' if same_sent else 'DIFFERENT'})"))
for t in ties11:
    print(f"  near tie, not a change: {t}")

BM25, dense, weighted, BM25+map, weighted+map, weighted+neighbours against evals/baselines/day-11-retrieval-v3.json: no changes (190 queries x 6: every rank, best score, section sent and metric; top 3 lists and sections sent identical)


Same labels, same snapshot, same rankings and contexts.

### 1.1 New held-out questions: `questions-v4`

Day 11 used v3's 68 test questions to decide, so they count as used now (`evals/README.md`, rules 4 and 8). [`evals/questions-v4.jsonl`](../evals/questions-v4.jsonl) keeps v3's 190 queries exactly (the 68 become dev; each note says so) and adds **76 new test questions**: 52 answerable (3 asked by maria about the managers' guide) and 24 not, a larger unanswerable share than before, because refusals are half of what an answer evaluation measures. They were written after the rules were frozen and committed with them, before any system ran on them; the pre-registration says what the author knew while writing them. Only questions get an end-to-end grade: a lookup (an exact value typed as the query) isn't asking for an answer.

In [5]:
manifest4 = json.loads(V4["manifest"].read_text(encoding="utf-8"))
queries4 = load_queries(V4["questions"])
problems4 = validate(queries4, chunks, manifest4)
v4_sha256 = hashlib.sha256(V4["questions"].read_bytes()).hexdigest()
if v4_sha256 != manifest4["questions_sha256"]:
    problems4.insert(0, f"{V4['questions'].name} changed but its manifest didn't")
by_qid = {q.id: q for q in queries4}
used = [q.id for q in queries3 if q.split == "test"]
# v4 keeps v3's 190 queries exactly (text, user, labels); only the used test questions changed split (and say so)
same190 = all(q4 == dataclasses.replace(q3, split=q4.split, note=q4.note) for q3, q4 in zip(queries3, queries4)) and \
    [q.id for q in queries4[:190] if q.split != "dev"] == [] and \
    all(by_qid[qid].note.endswith("so dev from v4 on)") for qid in used) and \
    all(q4.note == q3.note for q3, q4 in zip(queries3, queries4) if q3.id not in used)
NEW = [q for q in queries4 if q.split == "test"]
natural = [q for q in queries4 if q.kind == "question"]      # an answer is graded for questions, not pasted values
DEV = [q for q in natural if q.split == "dev"]
print(f"{len(queries4)} labelled queries (version {manifest4['version']}): v3's {len(queries3)} "
      f"({'identical, its ' + str(len(used)) + ' test questions now dev' if same190 else 'CHANGED'}) + "
      f"{len(NEW)} new test questions ({sum(q.answerable for q in NEW)} answerable, "
      f"{sum(not q.answerable for q in NEW)} not), same snapshot | label problems: {len(problems4)}"
      + "".join(f"\n  {p}" for p in problems4))
print(f"questions to grade end to end: dev {len(DEV)} ({sum(q.answerable for q in DEV)} answerable), "
      f"test {len(NEW)} ({sum(q.answerable for q in NEW)} answerable)")
assert not problems4 and same190 and manifest4["snapshot"] == manifest3["snapshot"]
assert {q.origin for q in NEW} == {"day12-new"} and all(q.kind == "question" for q in NEW)

266 labelled queries (version 4): v3's 190 (identical, its 68 test questions now dev) + 76 new test questions (52 answerable, 24 not), same snapshot | label problems: 0
questions to grade end to end: dev 160 (112 answerable), test 76 (52 answerable)


## 2. The current pipeline, and three failures that retrieval metrics can't see

The **current pipeline** is Day 11's end-to-end path, the no-change baseline today: the weighted fusion's top 3 as the context (Day 11's expansion wasn't adopted), Day 07's gate on the **dense** top score (0.62; a fused score is near 1 for every question by construction), Day 06's extractive reader at Day 07's threshold (0.63) with Day 06's names rule, and Day 06's citation check. It must give Day 11's published end-to-end numbers first. Then three recorded failures, each with the answer in, or right next to, the context.

In [6]:
cache = SentenceCache(embedder)            # Day 11: each sentence embedded once, whichever context it is in
READERS = {"current": ExtractiveReader(cache, min_score=READER_THRESHOLD),            # Day 06's names rule
           "names-v2": NamesReader(cache, unknown_names_v2, min_score=READER_THRESHOLD)}   # Day 10's (the candidate)


def sent_of(q) -> tuple[str, ...]:
    return tuple(cid for cid, _ in rankings["weighted"][q.id][:HITS])      # the current pipeline's context


def run(q, generator, sent=None):
    # The current pipeline, end to end: the weighted fusion's top 3 (or another context), Day 07's gate on the dense
    # top score, Day 06's budget, the generator, Day 06's citation check
    sent = sent_of(q) if sent is None else sent
    return answer_context(q.user, q.question, sent, by_cid, dict(rankings["weighted"][q.id]),
                          rankings["dense"][q.id][0][1], generator, gate=GATE)


def show(a, q=None) -> None:
    sources = [s.id for s in a.context.sources] if a.context else []
    print(f"{a.user}: {a.question}\n  sent: {', '.join(sources) or '(nothing: refused before reading)'}"
          + (f"\n  labelled: {', '.join(q.answer_chunks)}" if q is not None and q.answerable else "")
          + f"\n  [{a.status}] {a.text}\n  why: {a.reason}")


# Day 11's end-to-end description, reproduced: the top 3 on v2's 122 queries (Day 11's dev) and v3's test questions
DAY11_E2E = {"dev": (45, 6, 9, 21, 11), "v3 test": (24, 6, 22, 8, 8)}
v3_natural = [q for q in queries3 if q.kind == "question"]
reproduced_e2e = {}
for split, part in {"dev": [q for q in v3_natural if q.split == "dev"],
                    "v3 test": [q for q in v3_natural if q.split == "test"]}.items():
    o = [(q, outcome(run(q, READERS["current"]), q)) for q in part]
    a, u = Counter(x for q, x in o if q.answerable), Counter(x for q, x in o if not q.answerable)
    reproduced_e2e[split] = (a["correct"], a["wrong"], a["refused"], u["refused"], u["answered"])
print("Day 11's end-to-end numbers for the top 3 (answerable correct / wrong / refused, unanswerable refused / answered): "
      + " | ".join(f"{split} {reproduced_e2e[split]}" for split in DAY11_E2E)
      + (" -> reproduced" if reproduced_e2e == DAY11_E2E else " -> DIFFERENT"))
print()

# The three failures today starts from
DADS, LOST, PHONE, LINK, FIRST_DAY = (by_qid[i] for i in ("d01-01", "d03-13", "d10-a24", "d11-a40", "d11-a17"))
LUNA_DADS = ("Secondary caregivers receive 6 weeks of fully paid leave, and leave can be taken in up to two blocks "
             "within 12 months of the child's arrival. [parental-leave-policy#leave-entitlement]")   # Day 06's live run
print("1. Right numbers, wrong meaning: gpt-6-luna's reply to new dads on Day 06 (recorded), put through today's pipeline")
luna = run(DADS, FixedReply(LUNA_DADS))
show(luna, DADS)
print(f"  outcome: {outcome(luna, DADS)} (facts: {', '.join(DADS.facts)})\n")
print("2. The answer is in the context, and the reader copies another sentence")
show(run(LOST, READERS["current"]), LOST)
print(f"  outcome: {outcome(run(LOST, READERS['current']), LOST)} (facts: {', '.join(LOST.facts)})")
expanded = run(LINK, READERS["current"], neighbours_of(LINK).sent)        # Day 11's expansion found its answer section
show(expanded, LINK)
print(f"  outcome: {outcome(expanded, LINK)} (facts: {', '.join(LINK.facts)})\n")
print("3. A sentence's first word taken for a name")
show(run(FIRST_DAY, READERS["current"]), FIRST_DAY)

Day 11's end-to-end numbers for the top 3 (answerable correct / wrong / refused, unanswerable refused / answered): dev (45, 6, 9, 21, 11) | v3 test (24, 6, 22, 8, 8) -> reproduced

1. Right numbers, wrong meaning: gpt-6-luna's reply to new dads on Day 06 (recorded), put through today's pipeline
alice: How much time off do new dads get?
  sent: parental-leave-policy#returning-to-work, parental-leave-policy#who-is-eligible, parental-leave-policy#leave-entitlement
  labelled: parental-leave-policy#leave-entitlement
  [answered] Secondary caregivers receive 6 weeks of fully paid leave, and leave can be taken in up to two blocks within 12 months of the child's arrival. [parental-leave-policy#leave-entitlement]
  why: 
  outcome: wrong (facts: 16 weeks, 6 weeks)

2. The answer is in the context, and the reader copies another sentence
alice: How soon must I report a lost laptop?
  sent: travel-policy#safety-while-travelling, security-incident-reporting#what-not-to-do, security-incident-report

Day 11's numbers are reproduced. The three failures:

1. **Right numbers, wrong meaning** (Day 06): gpt-6-luna assumed a new dad is a secondary caregiver. Every number is in the cited source and the citation check passes; the policy is gender-neutral, and a primary-caregiver dad gets 16 weeks.
2. **The answer is in the context, and the reader copies another sentence** (Day 11): *How to report* is sent for the lost laptop, and with Day 11's expansion for the suspicious link; the reader picks the sentence closest to the question, which describes the event, not what to do.
3. **A sentence's first word taken for a name** (Days 07, 10, 11): Day 06's names rule skips only the question's first word, so "Where" in "It's my first day. Where do I pick up my laptop?" is an unknown name, and the question is refused with its answer in the first source.

## 3. The rubric

One verdict per answer, computed from the answer, the context it was given, the labels and the facts (all pre-registered):

| Part | Rule |
|---|---|
| **Outcome** | Day 07's `outcome()`: refused (abstained or blocked); for an answerable question, correct if every fact is a case-insensitive substring of the answer, else wrong; for an unanswerable one, answered |
| **Refused by** | the step that refused: gate, names rule, reader threshold or citation check |
| **Context** | was a labelled chunk among the sections sent? |
| **Citations** | does the answer cite a labelled chunk; citation precision = labelled chunks cited / chunks cited |
| **Grounded** | the citation check passed and every sentence has at least 0.75 of its content words in the sources it cites: a lexical stand-in for "supported", which can't tell meaning |
| **Stage** | where it went wrong: *gate*, *names rule* (a sentence's first word, or a name), *reader threshold*, *citation check*, *retrieval: answer not in the context*, *reader: answer in the context, not used*, *reader: right source, wrong part*; for an unanswerable question answered, *answered: nothing answers it* or *restricted to others* |

The **stage** is what makes the grade useful: it says which step to fix. Four replies to the new-dads question show why the parts have to be separate.

In [7]:
STOPWORDS = set("""a an and are as at be but by can could do does for from had has have how i if in into is it its
may me must my no not of on or our should so than that the their them then there these they this to up was we were
what when where which who why will with would you your""".split())
WORD = re.compile(r"[a-z0-9]+(?:[.,:/@-][a-z0-9]+)*")      # words, numbers, amounts, emails, URLs, times


def content_words(text: str) -> list[str]:
    return [w for w in WORD.findall(text.lower())
            if w not in STOPWORDS and (len(w) >= 3 or any(ch.isdigit() for ch in w))]


def support(claim: str, sources) -> float:
    # A lexical stand-in for "the cited sources support this sentence": the share of its content words they contain
    claim_words = content_words(claim)
    known = set(content_words(" ".join(s.text for s in sources)))
    return sum(w in known for w in claim_words) / len(claim_words) if claim_words else 1.0


@dataclass(frozen=True)
class Grade:
    outcome: str                      # correct | wrong | refused (answerable); refused | answered (unanswerable)
    refused_by: str | None            # gate | names rule | reader threshold | citation check | the model (live)
    label_in_context: bool | None     # answerable: was a labelled chunk sent (or about to be, for a gate refusal)?
    cites_label: bool | None          # answered and answerable: does it cite a labelled chunk?
    citation_precision: float | None  # answered and answerable: labelled chunks cited / chunks cited
    support: float | None             # answered: the lowest support of any of its sentences
    grounded: bool | None             # answered: passed the citation check and every sentence has support >= 0.75
    stage: str                        # "ok", or where it went wrong


def first_words(question: str) -> set[str]:
    # The first word of every sentence after the first (Day 06's names rule skips only the question's first word)
    return {s.split()[0] for s in SENTENCE.split(question.strip())[1:] if s.split()}


def grade(a, q, sent) -> Grade:
    # The rubric: one verdict per answer, from the answer, the context it was given, the labels and the facts
    out = outcome(a, q)
    refused_by = (why_refused(a) or "the model") if a.status != "answered" else None     # "the model": it abstained
    label_in = found(sent, q) if q.answerable else None
    cites_label = precision = sup = grounded = None
    if a.status == "answered":
        sources = {s.id: s for s in a.context.sources}
        cited = list(dict.fromkeys(i for c in a.check.claims for i in c.cited))
        sup = round(min(support(c.text, [sources[i] for i in c.cited if i in sources]) for c in a.check.claims), 3)
        grounded = a.check.passed and sup >= SUPPORT_MIN
        if q.answerable:
            cites_label = bool(set(cited) & set(q.answer_chunks))
            precision = round(sum(i in q.answer_chunks for i in cited) / len(cited), 3)
    if q.answerable:
        if out == "correct":
            stage = "ok"
        elif out == "refused":
            if refused_by == "names rule":
                names = set(a.reason.removeprefix("no source mentions ").split(", "))
                stage = "names rule: a sentence's first word" if names <= first_words(q.question) else "names rule: a name"
            else:
                stage = refused_by
        elif not label_in:
            stage = "retrieval: answer not in the context"
        elif cites_label:
            stage = "reader: right source, wrong part"
        else:
            stage = "reader: answer in the context, not used"
    else:
        stage = "ok" if out == "refused" else \
            "answered: restricted to others" if q.restricted_chunks else "answered: nothing answers it"
    return Grade(out, refused_by, label_in, cites_label, precision, sup, grounded, stage)


def value(out: str, cost: float = COST) -> float:
    # What an answer is worth to the person asking (pre-registered): correct +1, refused 0, wrong or answered -cost
    return {"correct": 1, "refused": 0, "wrong": -cost, "answered": -cost}[out]


PARAPHRASE = ("Primary caregivers get 16 weeks of paid leave and secondary caregivers get 6 weeks "
              "[parental-leave-policy#leave-entitlement].")                                     # written by hand
CHANGED_NUMBER = "Primary caregivers receive 18 weeks of fully paid leave [parental-leave-policy#leave-entitlement]."
print(f"{'new dads, four replies':<44}{'outcome':<10}{'check':<8}{'support':>8}{'grounded':>10}{'cites label':>13}  stage")
for label, reply in (("the extractive reader (today's pipeline)", READERS["current"]),
                     ("a faithful paraphrase (hand-written)", FixedReply(PARAPHRASE)),
                     ("gpt-6-luna on Day 06 (recorded)", FixedReply(LUNA_DADS)),
                     ("a changed number (hand-written)", FixedReply(CHANGED_NUMBER))):
    a = run(DADS, reply)
    g = grade(a, DADS, sent_of(DADS))
    check = "passed" if a.check and a.check.passed and not a.check.abstained else "blocked" if a.status == "blocked" else "-"
    print(f"{label:<44}{g.outcome:<10}{check:<8}{g.support if g.support is not None else '-':>8}"
          f"{str(g.grounded) if g.grounded is not None else '-':>10}{str(g.cites_label) if g.cites_label is not None else '-':>13}"
          f"  {g.stage}")

new dads, four replies                      outcome   check    support  grounded  cites label  stage
the extractive reader (today's pipeline)    wrong     passed       1.0      True        False  reader: answer in the context, not used
a faithful paraphrase (hand-written)        correct   passed     0.833      True         True  ok
gpt-6-luna on Day 06 (recorded)             wrong     passed       1.0      True         True  reader: right source, wrong part
a changed number (hand-written)             refused   blocked        -         -            -  citation check


**Grounded isn't correct.** The extractive reader's reply and gpt-6-luna's are both fully grounded (every word is in the cited source) and both wrong: one cites the wrong section (the reader didn't use the answer it was given), the other the right one with the wrong part of it (*right source, wrong part*: "right numbers, wrong meaning" in rubric terms). The faithful paraphrase is correct with 83% of its words in the source ("get" for "receive"). The changed number never reaches the user: the citation check blocks it. A groundedness metric alone (the "faithfulness" of many RAG tools) would pass three of these four.

### 3.1 The rubric on dev, the current pipeline (in-sample)

In [8]:
def answers_for(qs, pipeline: str) -> dict:
    return {q.id: run(q, READERS[pipeline]) for q in qs}


ANSWERS = {p: {} for p in PIPELINES}                              # pipeline -> question id -> RagAnswer
GRADES = {p: {} for p in PIPELINES}                               # pipeline -> question id -> Grade


def add_answers(qs) -> None:
    for p in PIPELINES:
        ANSWERS[p].update(answers_for(qs, p))
        GRADES[p].update({q.id: grade(ANSWERS[p][q.id], q, sent_of(q)) for q in qs})


def counts(p: str, qs) -> tuple:
    a = Counter(GRADES[p][q.id].outcome for q in qs if q.answerable)
    u = Counter(GRADES[p][q.id].outcome for q in qs if not q.answerable)
    return (a["correct"], a["wrong"], a["refused"], u["refused"], u["answered"],
            sum(value(GRADES[p][q.id].outcome) for q in qs))


def rubric_table(p: str, qs, label: str) -> None:
    answerable, unanswerable = [q for q in qs if q.answerable], [q for q in qs if not q.answerable]
    c = counts(p, qs)
    print(f"{label} ({len(answerable)} answerable, {len(unanswerable)} not):")
    for name, n, total in (("correct", c[0], len(answerable)), ("wrong", c[1], len(answerable)),
                           ("refused", c[2], len(answerable)), ("unanswerable refused", c[3], len(unanswerable)),
                           ("unanswerable answered", c[4], len(unanswerable))):
        low, high = wilson(n, total)
        print(f"  {name:<24}{n:>4} of {total:<4}{n / total:>5.0%} ({low:.0%} to {high:.0%})")
    answered = [GRADES[p][q.id] for q in qs if GRADES[p][q.id].outcome in ("correct", "wrong", "answered")]
    cited = [g for g in answered if g.cites_label is not None]
    print(f"  value at cost {COST}: {c[5]:g} | answered {len(answered)}: grounded {sum(g.grounded for g in answered)}, "
          f"lowest support {min((g.support for g in answered), default=1):.2f} | answerable answered {len(cited)}: cite a "
          f"labelled chunk {sum(g.cites_label for g in cited)}, mean citation precision "
          f"{sum(g.citation_precision for g in cited) / max(1, len(cited)):.2f}")
    stages = Counter(GRADES[p][q.id].stage for q in answerable if GRADES[p][q.id].stage != "ok")
    print("  answerable, not correct: " + ", ".join(f"{s} {n}" for s, n in stages.most_common()))
    labels = Counter(GRADES[p][q.id].label_in_context for q in answerable)
    print(f"  a labelled chunk was in the context for {labels[True]} of {len(answerable)} answerable questions")


add_answers(DEV)
rubric_table("current", DEV, "Dev, the current pipeline (in-sample)")
assert counts("current", DEV) == PREREGISTERED["dev"]["current"], "the dev values don't match the pre-registration"

Dev, the current pipeline (in-sample) (112 answerable, 48 not):
  correct                   69 of 112   62% (52% to 70%)
  wrong                     12 of 112   11% (6% to 18%)
  refused                   31 of 112   28% (20% to 37%)
  unanswerable refused      29 of 48    60% (46% to 73%)
  unanswerable answered     19 of 48    40% (27% to 54%)
  value at cost 2: 7 | answered 100: grounded 100, lowest support 1.00 | answerable answered 81: cite a labelled chunk 71, mean citation precision 0.88
  answerable, not correct: names rule: a sentence's first word 19, reader: answer in the context, not used 8, names rule: a name 6, gate 3, retrieval: answer not in the context 2, reader: right source, wrong part 2, reader threshold 2, citation check 1
  a labelled chunk was in the context for 107 of 112 answerable questions


Of the 112 answerable dev questions, 107 have a labelled chunk in the context, and 69 are answered correctly. The stages say where the rest go: **the names rule refuses 25** (19 for a sentence's first word), and **the reader loses 10** (8 with the answer in the context but another section copied, 2 with the right section and the wrong part of it). Retrieval loses 2. Every answer is grounded, because this reader copies sentences: groundedness can't separate its good answers from its bad ones. Of the 48 unanswerable questions, 19 get an answer: a sentence from a nearby policy presented as if it answered (*answered: nothing answers it*).

## 4. Grading the grader

An automatic grade is only as good as its agreement with a careful reading. The author read every answered (question, answer) pair, both pipelines, and graded it **correct** (a person asking gets the right answer and nothing misleading), **partly** (true and relevant but incomplete, or the right value among others without saying which applies) or **wrong**. The reviews are committed in [`evals/reviews/day-12-answers.jsonl`](../evals/reviews/day-12-answers.jsonl), each tied to the exact answer text by a hash: if an answer changes, its review is stale and must be redone. The dev answers were read during exploration, before the held-out run, and the grades written down after it; the reading isn't blind (the author knew the rubric's verdicts).

In [9]:
def answer_sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


reviews = [json.loads(line) for line in REVIEWS_FILE.read_text(encoding="utf-8").splitlines() if line.strip()] \
    if REVIEWS_FILE.is_file() else []
REVIEWED = {(r["id"], r["answer_sha256"]): r for r in reviews}


def agreement(qs, split: str) -> dict:
    # The rubric's verdict against the author's reading, for every distinct answered (question, answer) pair
    pairs = {(q.id, answer_sha(ANSWERS[p][q.id].text)): (q, p) for q in qs for p in PIPELINES
             if ANSWERS[p][q.id].status == "answered"}
    result = {"pairs": len(pairs), "unreviewed": [], "agree": 0, "over": [], "under": [], "partly": 0}
    for key, (q, p) in pairs.items():
        r = REVIEWED.get(key)
        if r is None or r["split"] != split:
            result["unreviewed"].append(key)
            continue
        auto = GRADES[p][q.id].outcome == "correct"
        result["partly"] += r["grade"] == "partly"
        if auto == (r["grade"] == "correct"):
            result["agree"] += 1
        elif auto:
            result["over"].append((q, p, r))     # the rubric says correct, the reading doesn't
        else:
            result["under"].append((q, p, r))    # the reading says correct, the rubric doesn't
    return result


def show_agreement(result: dict, label: str) -> None:
    reviewed = result["pairs"] - len(result["unreviewed"])
    print(f"{label}: {result['pairs']} distinct answered (question, answer) pairs, {reviewed} reviewed; the rubric "
          f"agrees with the reading on {result['agree']} (reading: correct, partly or wrong; 'partly' counts as not "
          f"correct, {result['partly']} of them)")
    for kind, items in (("the rubric says correct, the reading doesn't", result["over"]),
                        ("the reading says correct, the rubric doesn't", result["under"])):
        for q, p, r in items:
            print(f"  {kind}: {q.id} {q.question}\n    {ANSWERS[p][q.id].text[:150]}\n    reading: {r['grade']}: "
                  f"{r['note']}")
    if result["unreviewed"]:
        print(f"  not reviewed yet (or the answer changed since its review): "
              + ", ".join(qid for qid, _ in result["unreviewed"]))


DEV_AGREEMENT = agreement(DEV, "dev")
show_agreement(DEV_AGREEMENT, "Dev, both pipelines")
stale = [k for k, r in REVIEWED.items() if r["split"] == "dev" and
         not any((q.id, answer_sha(ANSWERS[p][q.id].text)) == k for q in DEV for p in PIPELINES)]
print(f"reviews of dev answers that no longer occur (stale): {len(stale)}")

Dev, both pipelines: 122 distinct answered (question, answer) pairs, 122 reviewed; the rubric agrees with the reading on 120 (reading: correct, partly or wrong; 'partly' counts as not correct, 4 of them)
  the rubric says correct, the reading doesn't: d07-a01 What is the hotel limit per night in Tokyo?
    City: London | Hotel per night: USD 250 | Meals per day: USD 75 [travel-policy#daily-limits-by-city]. City: Singapore | Hotel per night: USD 220 | Mea
    reading: partly: lists three rows, including Other cities (USD 180), without saying which applies to Tokyo
  the rubric says correct, the reading doesn't: d10-a01 What's the hotel budget per night in Berlin?
    City: London | Hotel per night: USD 250 | Meals per day: USD 75 [travel-policy#daily-limits-by-city]. City: Other cities | Hotel per night: USD 180 | 
    reading: partly: lists London and Other cities (USD 180) without saying which applies to Berlin
reviews of dev answers that no longer occur (stale): 0


The rubric agrees with the reading on 120 of 122 answers. Both disagreements are the same limit of fact containment: **a list of table rows that includes the right one** (Tokyo and Berlin get every row, including Other cities, without being told which applies). The grader is good here *because this reader copies the documents' words*; a model that paraphrases ("two weeks" for "14 days") would make containment fail the other way. So the agreement has to be measured again whenever the generator changes: section 10's live run is where that happens.

## 5. The candidate on dev: Day 10's names rule (in-sample)

Day 10's names rule skips every sentence's first word and knows a gazetteer place when a source mentions its type ("Use the Other cities row for any city not listed"). Day 10 measured it with refusals counted, not answers graded, and didn't adopt it (it had to lose nothing). Today it is compared **by answers**, question by question, with the value of each answer: correct +1, refused 0, wrong −2, an unanswerable question answered −2 (pre-registered). A changed outcome is set apart as a near tie if any call that decided either pipeline's answer was within 0.001.

In [10]:
def better_or_worse(q, a: str, b: str) -> int:
    # +1 if pipeline a's outcome is better for the person asking, -1 if worse, 0 if the same
    rank = {"correct": 2, "refused": 1, "wrong": 0} if q.answerable else {"refused": 1, "answered": 0}
    return (rank[GRADES[a][q.id].outcome] > rank[GRADES[b][q.id].outcome]) - \
        (rank[GRADES[a][q.id].outcome] < rank[GRADES[b][q.id].outcome])


def closest_call(q, p: str) -> tuple[float, str]:
    # The smallest gap that decided this answer: another CPU's rounding (gaps under TIE) could flip it
    a, reader = ANSWERS[p][q.id], READERS[p]
    calls = [(abs(rankings["dense"][q.id][0][1] - GATE), "gate"), (cutoff_gap(q, HITS), "which sections are sent")]
    if a.context is not None and a.reason.startswith("best sentence"):
        ranked = reader.scored_units(a.context)
        best, source = ranked[0][0], ranked[0][1].id
        calls.append((abs(best - reader.min_score), "reader threshold"))
        other = next((s for s, src, _ in ranked if src.id != source), None)
        if other is not None:
            calls.append((best - other, "which source is copied"))
        calls += [(abs(s - (best - reader.margin)), "which sentences are copied")
                  for s, src, _ in ranked[1:] if src.id == source]
    return min(calls)


def paired(qs, label: str) -> dict:
    changed = [q for q in qs if GRADES["names-v2"][q.id].outcome != GRADES["current"][q.id].outcome]
    near = [q for q in changed if min(closest_call(q, p)[0] for p in PIPELINES) < TIE]
    kept = [q for q in changed if q not in near]

    def tally(group):
        better = [q for q in group if better_or_worse(q, "names-v2", "current") > 0]
        worse = [q for q in group if better_or_worse(q, "names-v2", "current") < 0]
        delta = {cost: sum(value(GRADES["names-v2"][q.id].outcome, cost) - value(GRADES["current"][q.id].outcome, cost)
                           for q in group) for cost in (1, 2, 3)}
        return better, worse, delta

    better, worse, delta = tally(kept)
    every = tally(changed)
    print(f"{label}: Day 10's names rule against the current pipeline, {len(qs)} questions | outcome changed for "
          f"{len(changed)} | near ties set apart: {', '.join(q.id for q in near) or 'none'} | better {len(better)}, "
          f"worse {len(worse)}, sign test p = {sign_test(len(better), len(worse)):.2g} | change in value: "
          + ", ".join(f"cost {c}: {d:+g}" for c, d in delta.items()))
    if near:
        print(f"  counting the near ties too: better {len(every[0])}, worse {len(every[1])}, change in value: "
              + ", ".join(f"cost {c}: {d:+g}" for c, d in every[2].items()))
    for q in changed:
        g0, g1 = GRADES["current"][q.id], GRADES["names-v2"][q.id]
        tag = "near" if q in near else "better" if q in better else "worse"
        print(f"  {tag:<7}{q.id:<9}{g0.outcome:>8} ({g0.stage}) -> {g1.outcome} ({g1.stage}): {q.question[:64]}")
    return {"changed": [q.id for q in changed], "near": [q.id for q in near], "better": [q.id for q in better],
            "worse": [q.id for q in worse], "delta": delta, "p": sign_test(len(better), len(worse)),
            "every changed question": {"better": len(every[0]), "worse": len(every[1]), "delta": every[2]}}


rubric_table("names-v2", DEV, "Dev, the candidate (Day 10's names rule; in-sample)")
print()
DEV_PAIRED = paired(DEV, "Dev (in-sample)")
every = DEV_PAIRED["every changed question"]       # the pre-registration's dev line counts every changed question
assert counts("names-v2", DEV) == PREREGISTERED["dev"]["names-v2"] and \
    (every["better"], every["worse"], every["delta"]) == (PREREGISTERED["better"], PREREGISTERED["worse"],
                                                         PREREGISTERED["delta"]), \
    "the candidate's dev values don't match the pre-registration: stop and find out why"

Dev, the candidate (Day 10's names rule; in-sample) (112 answerable, 48 not):
  correct                   84 of 112   75% (66% to 82%)
  wrong                     18 of 112   16% (10% to 24%)
  refused                   10 of 112    9% (5% to 16%)
  unanswerable refused      28 of 48    58% (44% to 71%)
  unanswerable answered     20 of 48    42% (29% to 56%)
  value at cost 2: 8 | answered 122: grounded 122, lowest support 1.00 | answerable answered 102: cite a labelled chunk 88, mean citation precision 0.86
  answerable, not correct: reader: answer in the context, not used 10, reader: right source, wrong part 4, retrieval: answer not in the context 4, names rule: a name 3, gate 3, reader threshold 2, citation check 2
  a labelled chunk was in the context for 107 of 112 answerable questions



Dev (in-sample): Day 10's names rule against the current pipeline, 160 questions | outcome changed for 22 | near ties set apart: d10-a01 | better 14, worse 7, sign test p = 0.19 | change in value: cost 1: +7, cost 2: +0, cost 3: -7
  counting the near ties too: better 15, worse 7, change in value: cost 1: +8, cost 2: +1, cost 3: -6
  better d07-a01   refused (names rule: a name) -> correct (ok): What is the hotel limit per night in Tokyo?
  better d07-a11   refused (names rule: a sentence's first word) -> correct (ok): My laptop might have a virus. Should I switch it off?
  near   d10-a01   refused (names rule: a name) -> correct (ok): What's the hotel budget per night in Berlin?
  worse  d10-a02   refused (names rule: a name) -> wrong (reader: right source, wrong part): How much can I spend on food per day in Paris?
  better d10-a23   refused (names rule: a sentence's first word) -> correct (ok): I'm travelling abroad next month. Do I need to register the trip
  worse  d10-a24   refus

**Fixing the names rule exposes the steps after it.** It turns 22 refusals into answers: 15 correct, 6 wrong and one unanswerable question answered. Every wrong one fails downstream: the reader copies the wrong row or section (4), or retrieval never sent the answer (2: the stolen phone and the train ticket). The old rule was hiding those failures by refusing questions at random, for a capitalised word.

One changed outcome is a near tie: Berlin's answer copies the Other cities row because it is within the reader's margin by 0.0008; another CPU could drop it, and the answer would be wrong. With it set apart the candidate is better for 14 questions and worse for 7 (p = 0.19), and the value at cost 2 doesn't move (+0). **The pre-registration's dev line counts every changed question, near ties included** (15 better, +1 at cost 2): the notebook checks that line as committed, and reports both.

## 6. The held-out test: 76 new questions

The comparison the pre-registration made: both pipelines on the 76 new questions, graded by the rubric, question by question. First, retrieval on the new questions, as a description.

In [11]:
add_rankings(NEW)          # the new questions: nothing about them was used to set anything today
ANSWERABLE_NEW = [q for q in NEW if q.answerable]
print(f"{'v4 test (52 answerable)':<26}" + "".join(f"{'hit@' + str(k):>7}" for k in K_VALUES) + f"{'MRR@10':>8}"
      "   hit@3 rate, 95% interval")
for name in ("BM25", "dense", "weighted"):
    m = measure(rankings[name], ANSWERABLE_NEW)
    low, high = wilson(m["hit@3"], m["n"])
    print(f"{name:<26}" + "".join(f"{m['hit@' + str(k)]:>7}" for k in K_VALUES)
          + f"{m['MRR@10']:>8.2f}   {m['hit@3'] / m['n']:.0%} ({low:.0%} to {high:.0%})")
print()
add_answers(NEW)
for p in PIPELINES:
    rubric_table(p, NEW, f"v4 test, {'the current pipeline' if p == 'current' else 'the candidate'}")
    print()
TEST_PAIRED = paired(NEW, "v4 test (held-out)")

v4 test (52 answerable)     hit@1  hit@3  hit@5 hit@10  MRR@10   hit@3 rate, 95% interval
BM25                           43     49     49     52    0.89   94% (84% to 98%)
dense                          44     51     51     52    0.92   98% (90% to 100%)
weighted                       49     52     52     52    0.97   100% (93% to 100%)



v4 test, the current pipeline (52 answerable, 24 not):
  correct                   30 of 52    58% (44% to 70%)
  wrong                      3 of 52     6% (2% to 16%)
  refused                   19 of 52    37% (25% to 50%)
  unanswerable refused      14 of 24    58% (39% to 76%)
  unanswerable answered     10 of 24    42% (24% to 61%)
  value at cost 2: 4 | answered 43: grounded 43, lowest support 1.00 | answerable answered 33: cite a labelled chunk 31, mean citation precision 0.94
  answerable, not correct: names rule: a sentence's first word 12, names rule: a name 5, reader: answer in the context, not used 2, citation check 2, reader: right source, wrong part 1
  a labelled chunk was in the context for 52 of 52 answerable questions

v4 test, the candidate (52 answerable, 24 not):
  correct                   40 of 52    77% (64% to 86%)
  wrong                      5 of 52    10% (4% to 21%)
  refused                    7 of 52    13% (7% to 25%)
  unanswerable refused      11 of 24

v4 test (held-out): Day 10's names rule against the current pipeline, 76 questions | outcome changed for 15 | near ties set apart: d12-a01 | better 10, worse 4, sign test p = 0.18 | change in value: cost 1: +6, cost 2: +2, cost 3: -2
  counting the near ties too: better 10, worse 5, change in value: cost 1: +5, cost 2: +0, cost 3: -5
  near   d12-a01   refused (names rule: a sentence's first word) -> wrong (reader: answer in the context, not used): I'm cleaning up an old project folder. Can I delete invoices fro
  better d12-a03   refused (names rule: a sentence's first word) -> correct (ok): A customer asked us to delete their personal data a week ago. Is
  better d12-a04   refused (names rule: a sentence's first word) -> correct (ok): My manager approved my USD 600 claim. Does Finance need to appro
  better d12-a07   refused (names rule: a sentence's first word) -> correct (ok): I've got the receipts from a client visit. Where do I submit the
  better d12-a21   refused (names rule: a

**Retrieval has caught up with these documents**: the weighted fusion has every new answer in its top 3 (52 of 52), so on the held-out set every end-to-end failure happens after retrieval. The current pipeline answers 30 of 52 correctly; the names rule refuses 17 (12 for a sentence's first word).

With Day 10's names rule, 15 outcomes change: **10 refused → correct**, 1 refused → wrong (d12-a40: the reader copies *What not to do* for "Which email address should I report it to?"), and 3 unanswerable questions now get an answer (a commute by taxi, an allowance when leaving, a four-day week: all two-sentence questions the old rule refused by accident). One more (d12-a01) is a near tie and is set apart. Value at cost 2: **+2**. Both pipelines are worth 4 in total; the difference is the near tie (section 9).

### 6.1 Every new question, both pipelines

Every v4 test question with its outcome and stage under both pipelines (a stage is the diagnosis). No new question's answer is missing from the context, so there is nothing to diagnose for retrieval.

In [12]:
# Every v4 test question whose labelled chunk isn't in the weighted fusion's top 3 (written after reading each one).
# The assert fails if the set changes, so a new miss can't go unexplained.
CONTEXT_DIAGNOSES = {}
context_misses = [q for q in ANSWERABLE_NEW if not found(sent_of(q), q)]
assert {q.id for q in context_misses} == set(CONTEXT_DIAGNOSES), "every miss needs a diagnosis, and only misses"
print(f"v4 test questions whose labelled chunk isn't in the current pipeline's context: {len(context_misses)}")
for q in context_misses:
    category, why = CONTEXT_DIAGNOSES[q.id]
    print(f"  {q.id:<9}fused #{first_relevant(rankings['weighted'][q.id], q.answer_chunks) or '-':<4}{category:<16}"
          f"{q.question}\n{'':<29}{why}")
print()
print(f"{'v4 test question':<10}{'current pipeline':<44}{'candidate (Day 10 names rule)':<44}")
for q in NEW:
    g0, g1 = GRADES["current"][q.id], GRADES["names-v2"][q.id]
    print(f"{q.id:<10}{g0.outcome + ': ' + g0.stage:<44}{(g1.outcome + ': ' + g1.stage) if g1 != g0 else '(same)':<44}")

v4 test questions whose labelled chunk isn't in the current pipeline's context: 0

v4 test questioncurrent pipeline                            candidate (Day 10 names rule)               
d12-a01   refused: names rule: a sentence's first wordwrong: reader: answer in the context, not used
d12-a02   wrong: reader: right source, wrong part     (same)                                      
d12-a03   refused: names rule: a sentence's first wordcorrect: ok                                 
d12-a04   refused: names rule: a sentence's first wordcorrect: ok                                 
d12-a05   wrong: reader: answer in the context, not used(same)                                      
d12-a06   correct: ok                                 (same)                                      
d12-a07   refused: names rule: a sentence's first wordcorrect: ok                                 
d12-a08   correct: ok                                 (same)                                      
d12-a09   refuse

Two answerable questions are refused by both pipelines through the **citation check**: the reader copies a numbered list item ("1. Search the software catalogue…") and Day 06's `split_claims()` splits off "1." as a sentence without a citation. That is Day 07's recorded bug, now costing two held-out answers (and refusing one unanswerable question, d12-u22, by luck); it is regression case R21. Under the candidate the five remaining names-rule refusals are names the sections sent don't mention: Chrome and Slack (in no document), Madrid (not in the stand-in gazetteer), Singapore (the answer is in *Core hours*, which doesn't name a city) and Tokyo (a gazetteer city, but no section sent mentions a city).

### 6.2 The reading of the new answers (written after the run)

The same reading for every answered v4 test question, written after the held-out run and marked so. It describes the grader's agreement on new answers; it decides nothing.

In [13]:
TEST_AGREEMENT = agreement(NEW, "test")
show_agreement(TEST_AGREEMENT, "v4 test, both pipelines (the reading was written after the run)")

v4 test, both pipelines (the reading was written after the run): 58 distinct answered (question, answer) pairs, 58 reviewed; the rubric agrees with the reading on 57 (reading: correct, partly or wrong; 'partly' counts as not correct, 2 of them)
  the rubric says correct, the reading doesn't: d12-a49 What's the lowest hotel limit in the travel policy?
    City: London | Hotel per night: USD 250 | Meals per day: USD 75 [travel-policy#daily-limits-by-city]. City: Singapore | Hotel per night: USD 220 | Mea
    reading: partly: lists all three rows; the lowest limit (Other cities, USD 180) isn't picked out


57 of 58 agree; the one disagreement is the same limit again (d12-a49 lists every hotel row; the lowest isn't picked out).

## 7. Regression tests: every recorded failure, as a test

Each day of this series recorded failures. A failure that was fixed must stay fixed; a property that must always hold must be tested; and a failure that isn't fixed should be pinned exactly as it behaves, so that any change is noticed. [`evals/regressions.jsonl`](../evals/regressions.jsonl) lists 29 of them, one line each (the day it was recorded, the lesson that recorded it, what failed, and the check that pins it), and [`tests/test_regressions.py`](../tests/test_regressions.py) runs every one on every push, from the shared code only (about 20 seconds):

- **fixed** (9): new dads, the locked account, table rows cut from their header, the PDF's split word, exact values for dense retrieval, wine, the lost laptop, retired chunks that stayed in the index, the revised names rule;
- **guard** (13): quarantine of a document without permissions, filtering before ranking, no managers-only chunk for alice in any top 10, the atomic save, prompt escaping, the citation check's four blocks, no API key in an error, the gate, permissions end to end, exact values never rewritten, the expansion's off-switch, the checked neighbour lookup, stale labels reported;
- **known** (7): the stolen phone outside the top 3, the duplicate-id upsert, right numbers with the wrong meaning passing the citation check, the list-number split, Day 06's names rule and "Should", gazetteer gaps (Madrid), and the reader ignoring the answer in its context.

A **known** case passes while the failure still happens. If it starts passing, the test fails and says so: move it to **fixed** on purpose, and update the lesson that recorded it.

In [14]:
# The cases recorded up to Day 12 (R01 to R29). Later days add more; their own notebooks check those.
regressions = [r for r in (json.loads(line) for line in REGRESSIONS_FILE.read_text(encoding="utf-8").splitlines()
                           if line.strip()) if int(r["id"][1:]) <= 29]
by_status = Counter(r["status"] for r in regressions)
print(f"{len(regressions)} regression cases in {REGRESSIONS_FILE.relative_to(REPO_ROOT).as_posix()}: "
      + ", ".join(f"{n} {s}" for s, n in by_status.most_common()) + " (tests/test_regressions.py runs every one)")
for r in regressions:
    print(f"  {r['id']}  {r['status']:<6} {r['recorded']:<8} {r['what']}")
print()
# Three of them, computed here with today's objects: what each case pins
should = by_qid["d07-a11"]
sources = ANSWERS["current"][should.id].context.sources
print(f"{should.id}: Day 06's names rule on {should.question!r}: {unknown_names(should.question, sources)} | Day 10's: "
      f"{unknown_names_v2(should.question, sources)}")
print(f"d01-01: gpt-6-luna's recorded reply passes the citation check: {luna.check.passed} | the facts grade: "
      f"{outcome(luna, DADS)}")
leak = [q.id for q in queries4 if q.user == "alice" for name in ("BM25", "dense", "weighted")
        if any(by_cid[cid].allowed_groups == ("managers", "people-team") for cid, _ in rankings[name][q.id][:10])]
print(f"managers-only chunks in any of alice's top 10 (BM25, dense, weighted; {sum(q.user == 'alice' for q in queries4)} "
      f"queries): {len(leak)}")

29 regression cases in evals/regressions.jsonl: 13 guard, 9 fixed, 7 known (tests/test_regressions.py runs every one)
  R01  fixed  Day 01   'How much time off do new dads get?': BM25 shares no word with the answer (Day 01); the weighted fusion has it in its top 3
  R02  fixed  Day 01   'My account is locked, what now?': no stemming, so BM25 missed 'locks' (Day 01); the weighted fusion has Account lockout in its top 3
  R03  fixed  Day 03   fixed-size windows cut table rows off their header (Day 03); section chunks keep the header row, and the reader pairs every row with it
  R04  fixed  Day 02   the travel-policy PDF splits 'reim-/bursement' across a line; ingestion joins it
  R05  guard  Day 02   a document without a permission entry, and a scanned PDF without text, are quarantined, not indexed (fail closed)
  R06  fixed  Day 04   dense retrieval missed exact values (Day 04: 14 of 25); the weighted fusion puts all 30 lookups' answer first
  R07  fixed  Day 07   'Will Acme pay for win

Three of the cases, computed here with this notebook's objects: Day 06's names rule still flags "Should" and Day 10's doesn't (R22, R23); gpt-6-luna's recorded reply still passes the citation check and fails the facts (R17); and no managers-only chunk reaches any of alice's top 10 (R11).

## 8. What it costs

In [15]:
def timed(work, sample, rounds: int = 3) -> tuple[float, float]:
    samples = []
    for _ in range(rounds):
        for q in sample:
            started = time.perf_counter()
            work(q)
            samples.append((time.perf_counter() - started) * 1000)
    samples.sort()
    return statistics.median(samples), samples[max(0, math.ceil(0.95 * len(samples)) - 1)]


SAMPLE = natural[::8]          # 30 questions from every split
fresh = ExtractiveReader(embedder, min_score=READER_THRESHOLD)       # no sentence cache: what one request costs
timings = {"retrieve (BM25 + dense + fusion)": timed(lambda q: weighted(
               [(h.id, h.score) for h in bm25[q.user].search(q.question, len(chunks))],
               [(r.id, r.score) for r in rag.retrieve(q.user, q.question, len(chunks))], ALPHA), SAMPLE),
           "answer (gate, reader, check)": timed(lambda q: run(q, fresh), SAMPLE),
           "grade (the rubric)": timed(lambda q: grade(ANSWERS["current"][q.id], q, sent_of(q)), SAMPLE)}
for step, (median, p95) in timings.items():
    print(f"{step:<36} median {median:9.3f} ms   p95 {p95:9.3f} ms")

retrieve (BM25 + dense + fusion)     median    14.430 ms   p95    19.496 ms
answer (gate, reader, check)         median    99.407 ms   p95   156.142 ms
grade (the rubric)                   median     0.022 ms   p95     0.060 ms


Grading an answer with this rubric costs hundredths of a millisecond, against tens of milliseconds to retrieve and about a tenth of a second to answer (the reader embeds every sentence it might copy; 2-vCPU CPU, timings vary). A rule-based rubric is cheap enough to run on every question of every change. A model-based judge costs a model call per answer, and needs its own agreement check against people (LEARN.md, section 9).

## 9. Decision

The decision, and what it needed, were written down before any held-out question was scored ([`evals/preregistration/day-12.md`](../evals/preregistration/day-12.md)): later days use Day 10's names rule if, on the v4 test questions, its total value at cost 2 is higher than the current pipeline's, counting only changed outcomes that aren't near ties.

In [16]:
def with_and_without(q, p: str) -> list[str]:
    # A near tie on "which sentences are copied": the outcome with and without the borderline sentence(s)
    a, reader = ANSWERS[p][q.id], READERS[p]
    ranked = reader.scored_units(a.context)
    best, src = ranked[0][0], ranked[0][1]
    near_line = [u for s, x, u in ranked if x.id == src.id and abs(s - (best - reader.margin)) < TIE]
    within = [u for s, x, u in ranked if x.id == src.id and s >= best - reader.margin]
    results = []
    for units in (list(dict.fromkeys(within + near_line)), [u for u in within if u not in near_line]):
        units = sorted(units[:reader.max_sentences], key=answer_units(src).index)
        text = " ".join(f"{u.rstrip('.')} [{src.id}]." for u in units)
        results.append("correct" if all(f.lower() in text.lower() for f in q.facts) else "wrong")
    return results


gain = TEST_PAIRED["delta"][COST]
D1 = gain > 0
print(f"D1, Day 10's names rule in the current pipeline: on the {len(NEW)} v4 test questions the outcome changed for "
      f"{len(TEST_PAIRED['changed'])}, of which {len(TEST_PAIRED['near'])} near tie set apart: better for "
      f"{len(TEST_PAIRED['better'])}, worse for {len(TEST_PAIRED['worse'])}; the value at cost {COST} changes by "
      f"{gain:+g} -> " + ("adopted: later days use Day 10's names rule" if D1 else
                          "not adopted: Day 06's names rule stays; the candidate is recorded as measured"))
print("sensitivity (a description, not the decision): " + ", ".join(
    f"cost {c}: {d:+g} ({'adopted' if d > 0 else 'not adopted'})" for c, d in TEST_PAIRED["delta"].items()))
NEAR_TIES = {}
for split, result in (("dev", DEV_PAIRED), ("v4 test", TEST_PAIRED)):
    for qid in result["near"]:
        q = by_qid[qid]
        gap, what = min((closest_call(q, p), p) for p in PIPELINES)[0]
        p = min(PIPELINES, key=lambda x: closest_call(q, x)[0])
        alternatives = with_and_without(q, p) if what == "which sentences are copied" else None
        NEAR_TIES[qid] = (p, round(gap, 4), what, alternatives)
        counted = value(GRADES["names-v2"][qid].outcome) - value(GRADES["current"][qid].outcome)
        print(f"near tie set apart ({split}): {qid} {p}, {what} decided by {gap:.4f}; the outcome "
              + (f"with and without the borderline sentence: {' and '.join(alternatives)}" if alternatives else "")
              + f" | counting it would change the value at cost {COST} by {counted:+g}, to "
              f"{result['delta'][COST] + counted:+g}")

D1, Day 10's names rule in the current pipeline: on the 76 v4 test questions the outcome changed for 15, of which 1 near tie set apart: better for 10, worse for 4; the value at cost 2 changes by +2 -> adopted: later days use Day 10's names rule
sensitivity (a description, not the decision): cost 1: +6 (adopted), cost 2: +2 (adopted), cost 3: -2 (not adopted)
near tie set apart (dev): d10-a01 names-v2, which sentences are copied decided by 0.0008; the outcome with and without the borderline sentence: correct and wrong | counting it would change the value at cost 2 by +1, to +1
near tie set apart (v4 test): d12-a01 names-v2, which sentences are copied decided by 0.0004; the outcome with and without the borderline sentence: wrong and wrong | counting it would change the value at cost 2 by -2, to +0


**Adopted, by the smallest margin the rule allows**, and that margin depends on the near-tie rule. The rule sets apart any changed question with a call under 0.001, and d12-a01's close call only decides whether a second sentence is copied: its answer is wrong either way. Counted, it would make the change +0 (the two pipelines are worth 4 each), and the candidate would not be adopted. On dev the same rule set apart a near tie that really could flip (Berlin: correct with the borderline row, wrong without it). The rule was written too broadly: a better one would set a question apart only if flipping its close call changes the outcome. It isn't changed now, because changing a rule after seeing the held-out results, in the direction that flips the decision, is what pre-registration exists to prevent. So the decision is recorded as pre-registered, with this caveat beside it.

At cost 1 the candidate wins (+6); at cost 3 it loses (−2). The evidence underneath doesn't depend on the costs: **the names rule's fix turns refusals into answers that are right about two times in three** (10 of 15 on the held-out set, 15 of 22 on dev), and **every wrong one is a downstream failure** the old rule was hiding. Later days switch it on (`answering.py` doesn't change today), and the reader becomes the step to fix: it is where most answerable questions are now lost.

## 10. Optional: a live model, graded with the same rubric

Off by default. With `RAG_CHAT_*` in `.env` (Day 06's client, now in `chat.py`) and `DAY12_LIVE_MODEL=1`, the current pipeline's contexts go to the model instead of the extractive reader: the gate still refuses without calling it, and its replies pass Day 06's citation check. At most 40 requests: the six recorded failures, then the first 24 answerable and 10 unanswerable v4 test questions the gate lets through. One run of one model is an observation, not a measurement; its answers need the author's reading before the rubric's agreement on them means anything.

In [17]:
LIVE_MODEL = os.environ.get("DAY12_LIVE_MODEL") == "1"     # or set it to True here (a hosted model costs money)
MAX_LIVE_REQUESTS = 40
RECORDED = ["d01-01", "d03-13", "d10-a24", "d10-a12", "d11-a06", "d11-a40"]      # failures recorded on Days 01-11
gated = [q for q in NEW if rankings["dense"][q.id][0][1] >= GATE]             # the gate refuses the rest unread
LIVE_QUESTIONS = [by_qid[qid] for qid in RECORDED] + [q for q in gated if q.answerable][:24] + \
    [q for q in gated if not q.answerable][:10]
assert len(LIVE_QUESTIONS) <= MAX_LIVE_REQUESTS
messages = build_messages(run(DADS, READERS["current"]).context)
input_tokens = round(sum(len(m["content"]) for m in messages) / 4)   # rule of thumb: ~4 characters per token
price_in, price_out = PRICES["gpt-6-luna"]
print(f"live run: {len(LIVE_QUESTIONS)} questions ({len(RECORDED)} recorded failures, then v4 test questions the gate "
      f"lets through); gpt-6-luna at most USD {len(LIVE_QUESTIONS) * (input_tokens * price_in + 1000 * price_out) / 1e6:.4f}"
      f" (USD {price_in} / {price_out} per million tokens, checked 2026-11-11)")
LIVE = {}
if not LIVE_MODEL:
    print("Live model: off. Fill in RAG_CHAT_* in .env (see .env.example), then set LIVE_MODEL = True.")
else:
    live = chat_model_from_env(REPO_ROOT / ".env")
    if live is None:
        raise ValueError("Set RAG_CHAT_PROVIDER and RAG_CHAT_MODEL in .env (see .env.example)")
    print(f"Live model: {live.name} at {live.url}\n")
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    spent_in = spent_out = 0
    with (OUT_DIR / "live-run.jsonl").open("w", encoding="utf-8") as log:
        for q in LIVE_QUESTIONS:
            started = time.perf_counter()
            try:
                a = run(q, live)
            except GeneratorError as error:            # the model couldn't be reached: record it, carry on
                print(f"{q.id}: unavailable ({error})")
                log.write(json.dumps({"id": q.id, "status": "unavailable", "error": str(error)}) + "\n")
                continue
            seconds = time.perf_counter() - started
            g = grade(a, q, sent_of(q))
            LIVE[q.id] = (a, g)
            usage = a.reply.usage if a.reply else {}
            spent_in += usage.get("input_tokens") or 0
            spent_out += usage.get("output_tokens") or 0
            print(f"{q.id} [{a.status}] {g.outcome}: {g.stage} | reader: {GRADES['current'][q.id].outcome}\n"
                  f"  {q.question}\n  {a.reply.text if a.reply else a.text}")
            log.write(json.dumps({"id": q.id, "question": q.question, "status": a.status, "reason": a.reason,
                                  "model_reply": a.reply.text if a.reply else None, "grade": dataclasses.asdict(g),
                                  "reader": dataclasses.asdict(GRADES["current"][q.id]),
                                  "usage": usage, "seconds": round(seconds, 2)}, ensure_ascii=False) + "\n")
    answered = Counter(g.outcome for a, g in LIVE.values())
    print(f"\n{len(LIVE)} graded: {dict(answered)} | stages: "
          + ", ".join(f"{s} {n}" for s, n in Counter(g.stage for a, g in LIVE.values()).most_common()))
    if live.model in PRICES:
        cost = (spent_in * PRICES[live.model][0] + spent_out * PRICES[live.model][1]) / 1e6
        print(f"total: {spent_in} input and {spent_out} output tokens, about USD {cost:.5f} at the prices above")

live run: 40 questions (6 recorded failures, then v4 test questions the gate lets through); gpt-6-luna at most USD 0.0218 (USD 0.1 / 0.5 per million tokens, checked 2026-11-11)
Live model: off. Fill in RAG_CHAT_* in .env (see .env.example), then set LIVE_MODEL = True.


## 11. Save the run, compare with the accepted baseline

One run file: every question of v4 (236), both pipelines, with the answer, its citations, the rubric's verdict and its closest call; the metrics, the paired comparison and the decision. The accepted copy is committed in [`evals/baselines/`](../evals/baselines/) as `day-12-answers-v4.json`, the first answer baseline: later days compare their answers with it by `compare_answers()`, which sets apart a change whose closest call is under 0.001 as CPU rounding.

In [18]:
DATASETS = {4: {"questions": V4["questions"].relative_to(REPO_ROOT).as_posix(), "version": manifest4["version"],
                "sha256": v4_sha256, "snapshot": manifest4["snapshot"]["fingerprint"]}}
PIPELINE_SETTINGS = {
    "current": {"retriever": "weighted fusion (Day 08, alpha_dense 0.7) of BM25 and dense, each permission-filtered",
                "context": f"the weighted fusion's top {HITS}", "budget_words": BUDGET,
                "gate": {"on": "the dense top score", "value": GATE, "from": "Day 07"},
                "reader": {"code": "rag_from_scratch.answering.ExtractiveReader (Day 06)", "min_score": READER_THRESHOLD,
                           "names_rule": "Day 06's unknown_names"},
                "check": "rag_from_scratch.answering.check_citations (Day 06)"},
    "names-v2": {"same as": "current", "reader": {"code": "rag_from_scratch.queries.NamesReader (Day 10)",
                                                   "min_score": READER_THRESHOLD,
                                                   "names_rule": "Day 10's unknown_names_v2"}}}
RUBRIC = {"outcome": "Day 07's outcome(): correct if every fact is a case-insensitive substring of the answer",
          "grounded": f"the citation check passed and every sentence has support >= {SUPPORT_MIN}",
          "support": "the share of a sentence's content words found in the sources it cites",
          "value": {"correct": 1, "refused": 0, "wrong": -COST, "answered (unanswerable)": -COST},
          "near_tie": f"an answer whose closest call is under {TIE}",
          "stages": sorted({g.stage for p in PIPELINES for g in GRADES[p].values()})}


def answer_entry(q, p: str) -> dict:
    a, g = ANSWERS[p][q.id], GRADES[p][q.id]
    gap, what = closest_call(q, p)
    return {"status": a.status, "text": a.text, "reason": a.reason,
            "cited": list(dict.fromkeys(i for c in a.check.claims for i in c.cited)) if a.check else [],
            **dataclasses.asdict(g), "closest_call": [round(gap, 4) if gap != math.inf else None, what]}


def metrics_for(qs) -> dict:
    return {p: dict(zip(("correct", "wrong", "refused", "unanswerable refused", "unanswerable answered",
                         f"value at cost {COST}"), counts(p, qs))) for p in PIPELINES}


run_file_data = {
    "name": "Day 12 answer baseline (questions v4)", "dataset": DATASETS[4], "pipelines": PIPELINE_SETTINGS,
    "rubric": RUBRIC,
    "metrics": {"dev": metrics_for(DEV), "v4 test": metrics_for(NEW)},
    "questions": {q.id: {"split": q.split,
                         "retrieval": {"top3": list(sent_of(q)), "dense_top": round(rankings["dense"][q.id][0][1], 4),
                                       "first_relevant": first_relevant(rankings["weighted"][q.id], q.answer_chunks)
                                       if q.answerable else None,
                                       "margin": round(cutoff_gap(q, HITS), 4)},
                         **{p: answer_entry(q, p) for p in PIPELINES}} for q in natural},
    "context_misses": [{"id": q.id, "first_relevant": first_relevant(rankings["weighted"][q.id], q.answer_chunks),
                        "category": CONTEXT_DIAGNOSES[q.id][0], "why": CONTEXT_DIAGNOSES[q.id][1]} for q in context_misses],
    "decisions": {"preregistration": PREREGISTRATION.relative_to(REPO_ROOT).as_posix(),
                  "dev": {k: DEV_PAIRED[k] for k in ("better", "worse", "near")}
                  | {"delta": {str(c): d for c, d in DEV_PAIRED["delta"].items()}},
                  "v4 test": {k: TEST_PAIRED[k] for k in ("better", "worse", "near")}
                  | {"delta": {str(c): d for c, d in TEST_PAIRED["delta"].items()}},
                  "D1 Day 10's names rule in the current pipeline": "adopted" if D1 else "not adopted"},
    "reviews": {"file": REVIEWS_FILE.relative_to(REPO_ROOT).as_posix(),
                "dev": {k: (len(v) if isinstance(v, list) else v) for k, v in DEV_AGREEMENT.items()},
                "v4 test": {k: (len(v) if isinstance(v, list) else v) for k, v in TEST_AGREEMENT.items()}}}


GRADE_FIELDS = tuple(f.name for f in dataclasses.fields(Grade))       # outcome, refused_by, ..., stage


def compare_answers(new: dict, old: dict) -> tuple[list[str], list[str]]:
    # What changed since the accepted answer baseline: (changes, near ties). Statuses, answer texts, citations and every
    # field of the verdict must match exactly; a change where either run's closest call is under TIE is CPU rounding.
    if new["dataset"] != old["dataset"]:
        return ["different labels or a different snapshot: these two runs can't be compared"], []
    changes = ["pipeline settings or the rubric changed"] if (new["pipelines"], new["rubric"]) != \
        (old["pipelines"], old["rubric"]) else []
    ties = []
    for qid, now in new["questions"].items():
        then = old["questions"].get(qid)
        if then is None:
            changes.append(f"{qid}: not in the baseline")
            continue
        for p in PIPELINES:
            fields = [f for f in ("status", "text", "cited", *GRADE_FIELDS) if now[p][f] != then[p][f]]
            if fields:
                moved = f"{qid} {p}: {', '.join(fields)} changed ({then[p]['outcome']} -> {now[p]['outcome']})"
                calls = [x[p]["closest_call"][0] for x in (now, then) if x[p]["closest_call"][0] is not None]
                (ties if calls and min(calls) < TIE else changes).append(moved)
    if new["metrics"] != old["metrics"]:
        (ties if ties and not changes else changes).append("metrics changed")
    if new["decisions"] != old["decisions"]:
        (ties if ties and not changes else changes).append("decisions changed")
    return changes, ties


OUT_DIR.mkdir(parents=True, exist_ok=True)
run_file = OUT_DIR / BASELINE_FILE.name
run_file.write_text(json.dumps(run_file_data, indent=1, ensure_ascii=False) + "\n", encoding="utf-8", newline="\n")
print(f"this run: {run_file.relative_to(REPO_ROOT).as_posix()} ({len(natural)} questions x {len(PIPELINES)} pipelines)")
if BASELINE_FILE.is_file():
    changes, ties = compare_answers(run_file_data, json.loads(BASELINE_FILE.read_text(encoding="utf-8")))
    print(f"compared with {BASELINE_FILE.relative_to(REPO_ROOT).as_posix()}: "
          + (f"{len(changes)} changes" + "".join(f"\n  {c}" for c in changes) if changes else
             f"no changes ({len(natural)} questions x {len(PIPELINES)} pipelines: status, outcome, stage, citations, "
             "answer text; metrics and decisions)"))
    for t in ties:
        print(f"  near tie, not a change: {t}")
else:
    changes, ties = None, []
    print(f"no accepted baseline yet: review {run_file.name}, then copy it to "
          f"{BASELINE_FILE.relative_to(REPO_ROOT).as_posix()} and commit it")
closest = sorted((closest_call(q, p)[0], q.id, p, closest_call(q, p)[1]) for q in natural for p in PIPELINES)
print("closest calls: " + ", ".join(f"{qid} {p} {gap:.4f} ({what})" for gap, qid, p, what in closest[:4]))

this run: outputs/day-12/day-12-answers-v4.json (236 questions x 2 pipelines)
compared with evals/baselines/day-12-answers-v4.json: no changes (236 questions x 2 pipelines: status, outcome, stage, citations, answer text; metrics and decisions)


closest calls: d10-u06 current 0.0001 (which sections are sent), d10-u06 names-v2 0.0001 (which sections are sent), d12-a01 names-v2 0.0004 (which sentences are copied), d10-a06 current 0.0005 (which source is copied)


## 12. Sanity checks

Each check pins a result quoted in this notebook or in LEARN.md. Ids, outcomes and counts are compared exactly; timings only for their shape.

In [19]:
def ids(qs) -> list[str]:
    return [q.id for q in qs]


# The same inputs as Day 11 (section 1), and the new questions
assert problems3 == [] and changes11 == [] and same_sent and len(queries3) == 190 and len(chunks) == 44
assert reproduced_e2e == DAY11_E2E
assert problems4 == [] and same190 and len(queries4) == 266 and len(NEW) == 76 and len(ANSWERABLE_NEW) == 52
assert manifest4["counts"]["dev"] == 190 and manifest4["counts"]["test"] == 76 and manifest4["previous"]["version"] == 3
assert len(DEV) == 160 and sum(q.answerable for q in DEV) == 112

# The three failures (section 2) and the rubric's four replies (section 3)
assert luna.status == "answered" and luna.check.passed and outcome(luna, DADS) == "wrong"
lost = ANSWERS["current"][LOST.id]
assert found(sent_of(LOST), LOST) and outcome(lost, LOST) == "wrong" and \
    [c for claim in lost.check.claims for c in claim.cited] == ["travel-policy#safety-while-travelling"]
assert found(neighbours_of(LINK).sent, LINK) and outcome(expanded, LINK) == "wrong" and \
    expanded.text.startswith("Do not try to investigate the incident yourself")
assert ANSWERS["current"][FIRST_DAY.id].reason == "no source mentions Where" and found(sent_of(FIRST_DAY), FIRST_DAY)
replies = {label: grade(run(DADS, r), DADS, sent_of(DADS)) for label, r in (
    ("reader", READERS["current"]), ("paraphrase", FixedReply(PARAPHRASE)), ("luna", FixedReply(LUNA_DADS)),
    ("changed", FixedReply(CHANGED_NUMBER)))}
assert [g.stage for g in replies.values()] == ["reader: answer in the context, not used", "ok",
                                               "reader: right source, wrong part", "citation check"]
assert replies["luna"].grounded and replies["luna"].cites_label and replies["reader"].grounded and \
    not replies["reader"].cites_label and replies["paraphrase"].support == 0.833 and replies["paraphrase"].grounded
# Support and citation precision count only the cited sources: the reader's sentence cited to the labelled section is
# ungrounded, and the paraphrase citing a second, unlabelled section is half precise (both hand-written, not shown above)
miscited = grade(run(DADS, FixedReply("You may return on a phased schedule of reduced hours for up to four weeks without "
                                      f"a change in pay [{DADS.answer_chunks[0]}].")), DADS, sent_of(DADS))
assert miscited.outcome == "wrong" and miscited.cites_label and not miscited.grounded and miscited.support < 0.5
second = next(cid for cid in sent_of(DADS) if cid not in DADS.answer_chunks)
both = grade(run(DADS, FixedReply(PARAPHRASE.replace("].", f"] [{second}]."))), DADS, sent_of(DADS))
assert both.outcome == "correct" and both.grounded and both.citation_precision == 0.5
assert content_words("Go to the IT desk by 9 am: USD 1,000, it@acme.example") == ["desk", "9", "usd", "1,000",
                                                                                  "it@acme.example"]
assert first_words("It's my first day. Where do I pick up my laptop?") == {"Where"}

# Dev (sections 3.1, 4 and 5): the pre-registered values, the stages, the reading
assert counts("current", DEV) == PREREGISTERED["dev"]["current"] and counts("names-v2", DEV) == PREREGISTERED["dev"]["names-v2"]
assert Counter(GRADES["current"][q.id].stage for q in DEV if q.answerable and GRADES["current"][q.id].stage != "ok") == \
    {"names rule: a sentence's first word": 19, "reader: answer in the context, not used": 8, "names rule: a name": 6,
     "gate": 3, "retrieval: answer not in the context": 2, "reader: right source, wrong part": 2, "reader threshold": 2,
     "citation check": 1}
assert sum(GRADES["current"][q.id].label_in_context for q in DEV if q.answerable) == 107
assert all(GRADES[p][q.id].grounded for p in PIPELINES for q in natural if GRADES[p][q.id].grounded is not None)
assert (DEV_AGREEMENT["pairs"], DEV_AGREEMENT["agree"], DEV_AGREEMENT["partly"], DEV_AGREEMENT["unreviewed"]) == (122, 120, 4, [])
assert [q.id for q, p, r in DEV_AGREEMENT["over"]] == ["d07-a01", "d10-a01"] and DEV_AGREEMENT["under"] == [] and stale == []
assert DEV_PAIRED["near"] == ["d10-a01"] and (len(DEV_PAIRED["better"]), len(DEV_PAIRED["worse"])) == (14, 7)
assert DEV_PAIRED["delta"] == {1: 7, 2: 0, 3: -7} and DEV_PAIRED["worse"] == [
    "d10-a02", "d10-a24", "d10-u12", "d11-a06", "d11-a41", "d11-a47", "d11-a52"]

# The held-out questions (section 6)
assert [tuple(measure(rankings[n], ANSWERABLE_NEW)[f"hit@{k}"] for k in K_VALUES) for n in ("BM25", "dense", "weighted")] \
    == [(43, 49, 49, 52), (44, 51, 51, 52), (49, 52, 52, 52)]
assert counts("current", NEW) == (30, 3, 19, 14, 10, 4) and counts("names-v2", NEW) == (40, 5, 7, 11, 13, 4)
assert TEST_PAIRED["near"] == ["d12-a01"] and TEST_PAIRED["worse"] == ["d12-a40", "d12-u01", "d12-u06", "d12-u15"]
assert TEST_PAIRED["better"] == ["d12-a03", "d12-a04", "d12-a07", "d12-a21", "d12-a23", "d12-a28", "d12-a37", "d12-a43",
                                 "d12-a46", "d12-a52"]
assert TEST_PAIRED["delta"] == {1: 6, 2: 2, 3: -2} and TEST_PAIRED["every changed question"]["delta"] == {1: 5, 2: 0, 3: -5}
assert context_misses == [] and CONTEXT_DIAGNOSES == {}
assert Counter(GRADES["current"][q.id].stage for q in ANSWERABLE_NEW if GRADES["current"][q.id].stage != "ok") == \
    {"names rule: a sentence's first word": 12, "names rule: a name": 5, "reader: answer in the context, not used": 2,
     "citation check": 2, "reader: right source, wrong part": 1}
assert [q.id for q in ANSWERABLE_NEW if GRADES["current"][q.id].stage == "citation check"] == ["d12-a09", "d12-a12"]
assert [q.id for q in ANSWERABLE_NEW if GRADES["names-v2"][q.id].stage == "names rule: a name"] == \
    ["d12-a11", "d12-a14", "d12-a34", "d12-a44", "d12-a48"]
assert (TEST_AGREEMENT["pairs"], TEST_AGREEMENT["agree"], TEST_AGREEMENT["partly"], TEST_AGREEMENT["unreviewed"]) == (58, 57, 2, [])
assert [q.id for q, p, r in TEST_AGREEMENT["over"]] == ["d12-a49"] and TEST_AGREEMENT["under"] == []

# Regression cases (section 7), cost (section 8), decision (section 9), the live run (section 10)
assert len(regressions) == 29 and by_status == {"guard": 13, "fixed": 9, "known": 7} and leak == []
assert timings["grade (the rubric)"][0] < timings["answer (gate, reader, check)"][0] / 100
assert D1 and gain == 2 and NEAR_TIES["d12-a01"][2:] == ("which sentences are copied", ["wrong", "wrong"])
assert NEAR_TIES["d10-a01"][2:] == ("which sentences are copied", ["correct", "wrong"])
assert len(LIVE_QUESTIONS) == 40 and ids(LIVE_QUESTIONS[:6]) == RECORDED and (LIVE_MODEL or LIVE == {})

# The run (section 11)
saved = json.loads(run_file.read_text(encoding="utf-8"))
assert saved["dataset"]["version"] == 4 and len(saved["questions"]) == 236 and saved["context_misses"] == []
assert saved["decisions"]["D1 Day 10's names rule in the current pipeline"] == "adopted"
assert changes is None or changes == [], changes            # near ties (CPU rounding) are allowed
if BASELINE_FILE.is_file():                                 # and the same answers as the accepted run, except near ties
    accepted = json.loads(BASELINE_FILE.read_text(encoding="utf-8"))
    steady = [(qid, p) for qid in accepted["questions"] for p in PIPELINES
              if min(accepted["questions"][qid][p]["closest_call"][0], saved["questions"][qid][p]["closest_call"][0]) >= TIE]
    assert all(saved["questions"][qid][p]["text"] == accepted["questions"][qid][p]["text"] for qid, p in steady)

print("All Day 12 sanity checks passed.")

All Day 12 sanity checks passed.


## 13. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing anything. Today's test split has been used to decide, so treat everything here as dev.

1. **Change the cost.** Set `COST = 1` and `COST = 3` in section 0 and rerun. Which outcomes and stages change? (None: only the values do.) At what cost does the decision flip, and what does that say about who should choose the cost?
2. **A stricter grader.** Add a rule to `grade()`: an answer that copies more than one table row is *partly* correct at best. How many dev and test answers does it change, and does the rubric's agreement with the reading improve?
3. **Fix the list-number split.** Change a copy of `split_claims()` so that a leading "1." stays with its sentence. Which regression case starts failing, and what would you change in `evals/regressions.jsonl`?
4. **A narrower near-tie rule.** Write the rule section 9 suggests: set a changed question apart only if flipping its close call changes the outcome. What does the decision become? Why mustn't you adopt it for today's decision?
5. **Grade a paraphrase.** Write three faithful paraphrases of correct dev answers ("14 days" as "two weeks", "five" as "5"). Which does the rubric fail, and how would you change the facts or the grader?

LEARN.md's exercise asks you to write a regression test for a failure you find yourself.